# Analyse Kiprix — Groupe 2
Notebook d'exploration des données scrapées depuis kiprix.com.

In [ ]:
from pathlib import Path
import pandas as pd
import re

data_path = Path('../data/raw/kiprix_mq.json')
if not data_path.exists():
    # Essayer la Guadeloupe si Martinique n'existe pas
    data_path = Path('../data/raw/kiprix_gp.json')
    
if data_path.exists():
    df = pd.read_json(data_path)
    print(f'Produits chargés depuis {data_path.name} : {len(df)}')
    display(df.head())
else:
    print("Aucun fichier de données trouvé. Lancez d'abord le scraping.")
    df = pd.DataFrame()

### 1. Nettoyage des données
Transformation des prix et des pourcentages en valeurs numériques.

In [ ]:
if not df.empty:
    def parse_diff(value):
        match = re.search(r'([+-]?\s*\d+[\d\s.,]*)\s*%', str(value))
        if not match:
            return None
        cleaned = match.group(1).replace(' ', '').replace('\u00a0', '').replace(',', '.')
        try:
            return float(cleaned)
        except ValueError:
            return None

    df['difference_numeric'] = df['difference'].apply(parse_diff)
    df['price_france_num'] = pd.to_numeric(
        df['price_france'].astype(str).str.replace('€', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce'
    )
    df['price_dom_num'] = pd.to_numeric(
        df['price_dom'].astype(str).str.replace('€', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce'
    )

    display(df[['difference_numeric', 'price_france_num', 'price_dom_num']].describe())

### 2. Statistiques globales

In [ ]:
if not df.empty:
    summary = {
        'total_produits': int(len(df)),
    }
    if 'difference_numeric' in df.columns and df['difference_numeric'].notna().any():
        summary.update({
            'moyenne_ecart_pct': float(df['difference_numeric'].dropna().mean()),
            'mediane_ecart_pct': float(df['difference_numeric'].dropna().median()),
            'max_ecart_pct': float(df['difference_numeric'].dropna().max()),
            'min_ecart_pct': float(df['difference_numeric'].dropna().min()),
        })
    display(summary)

### 3. Top 5 des produits les plus chers en DOM

In [ ]:
if not df.empty and 'price_dom_num' in df.columns:
    top5_dom = df.dropna(subset=['price_dom_num']).nlargest(5, 'price_dom_num')[['name', 'territory_name', 'price_dom', 'difference']]
    display(top5_dom)

### 4. Visualisations

In [ ]:
if not df.empty and 'difference_numeric' in df.columns:
    try:
        import matplotlib.pyplot as plt

        ax = df['difference_numeric'].dropna().plot(kind='hist', bins=20, title='Distribution des écarts de prix (%)', figsize=(8, 4), color='teal')
        ax.set_xlabel('Écart (%)')
        plt.show()
    except ModuleNotFoundError:
        print('matplotlib non installé. Installe-le avec: pip install matplotlib')